# 🚀 Быстрый старт: Использование NLP SDK

Этот ноутбук демонстрирует, как использовать класс `TextClassifier` для быстрого запуска модели машинного обучения в 3 строчки кода. Фасад автоматически подтягивает конфигурацию, настраивает устройство (CPU/GPU) и загружает веса.

В этом примере мы тестируем модель на задаче бинарной классификации (например, определение спама).

In [ ]:
# Настраиваем пути, чтобы ноутбук видел модули из папки src
import sys
from pathlib import Path

project_root = str(Path.cwd().parent)
if project_root not in sys.path:
    sys.path.append(project_root)

# Настройка логирования для красивого вывода в ноутбуке
import logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

## Инициализация классификатора

При инициализации SDK:
1. Автоматически парсится `configs/main.yaml`
2. Загружается токенизатор и базовая модель (`HFModelBuilder`)
3. Переносится на GPU (если доступно)

*Опционально:* Если у вас есть обученные веса, вы можете передать параметры `checkpoint_path` и `hf_repo_id` для автоматического скачивания.

In [ ]:
from src.sdk.inference import NLPPipeline

# Создаем инстанс. 
# Если вы уже обучили модель и залили на HF: 
# classifier = TextClassifier(hf_repo_id="your_name/spam-model", checkpoint_path="best.ckpt")

# Для классификации
spam_pipe = NLPPipeline('spam')
results = spam_pipe("Вам одобрен кредит!") 
# Вернет: [{'label_id': 1, 'confidence': 0.99}]

# Для генерации 
#gen_pipe = NLPPipeline('script_generator')
# text = gen_pipe("Напиши введение для новостного видео", max_new_tokens=150)
# Вернет: ["Всем привет! В сегодняшнем выпуске мы разберем..."]

## Запуск Инференса

Мы можем передавать как одиночные строки, так и батчи (списки строк). Метод `predict` возвращает структурированный словарь с вероятностями и предсказанным классом.

In [ ]:
# Тестовые данные (имитация входящих сообщений)
sample_messages = [
    "Здравствуйте, подскажите статус моей заявки №12345?",
    "Вам одобрен займ! Перейдите по ссылке http://scam-link.com чтобы забрать 100000 рублей прямо сейчас!!!",
    "Прошу направить справку 2-НДФЛ за 2023 год на эту почту."
]

# Инференс
results = classifier.predict(sample_messages)

# Красивый вывод результатов
for msg, res in zip(sample_messages, results):
    print("-" * 50)
    print(f"Текст: {msg}")
    print(f"Класс (ID): {res['label_id']}")
    print(f"Уверенность: {res['confidence'] * 100:.2f}%")